# Tiki Plan B — Qwen3 Back-Translation (RTX Pro 6000)

- Input: Kaggle Dataset `ntquang0410/tiki-plan-a-bt-input`
- Chỉ dịch route `mt` (29.446 cache entry)
- Checkpoint/resume theo `cache_key`
- Hỗ trợ Qwen3-0.6B / 1.7B / 4B / 8B
- Output mỗi model vào `/kaggle/working/plan_b_outputs/<model>/`


In [ ]:
import os, sys, json, time, re, math
from pathlib import Path

import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('vram_gb', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise RuntimeError('GPU chưa available; notebook này cần GPU.')


In [ ]:
# Competition notebook: không có internet.
# Dùng transformers/kagglehub có sẵn trong image + model_sources đã attach.
import transformers, kagglehub
print('transformers', transformers.__version__)
print('kagglehub', getattr(kagglehub, '__version__', 'installed'))


In [ ]:
import glob
import pandas as pd
from datasets import Dataset

CONFIG_FILES = {
    'qwen3-0.6b': 'qwen3_0_6b.json',
    'qwen3-1.7b': 'qwen3_1_7b.json',
    'qwen3-4b':   'qwen3_4b.json',
    'qwen3-8b':   'qwen3_8b.json',
}
MODEL_SOURCES = {
    'qwen3-0.6b': 'qwen-lm/qwen-3/Transformers/0.6b/1',
    'qwen3-1.7b': 'qwen-lm/qwen-3/Transformers/1.7b/1',
    'qwen3-4b':   'qwen-lm/qwen-3/Transformers/4b/1',
    'qwen3-8b':   'qwen-lm/qwen-3/Transformers/8b/1',
}

def find_dataset_dir():
    candidates = [
        Path('/kaggle/input/datasets/ntquang0410/tiki-plan-a-bt-input'),
        Path('/kaggle/input/tiki-plan-a-bt-input'),
    ]
    for base in candidates:
        if (base / 'tiki_bt_input.parquet').exists():
            return base
    matches = glob.glob('/kaggle/input/**/tiki_bt_input.parquet', recursive=True)
    if matches:
        return Path(matches[0]).parent
    raise FileNotFoundError('Không thấy tiki_bt_input.parquet trong /kaggle/input')

DATASET = find_dataset_dir()

# Smoke mặc định: 0.6B, 200 entry. Full: đổi LIMIT=None và bật list 4 model.
MODEL_KEYS_TO_RUN = ['qwen3-0.6b']
# MODEL_KEYS_TO_RUN = ['qwen3-0.6b', 'qwen3-1.7b', 'qwen3-4b', 'qwen3-8b']
LIMIT = 200
PROMPT_VERSION = 'plan_b_rtx_v1'
OUTPUT_ROOT = Path('/kaggle/working/plan_b_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_EVERY = 50

print('DATASET', DATASET)
print('available configs', sorted(p.name for p in DATASET.glob('*.json')))
print('OUTPUT_ROOT', OUTPUT_ROOT)
print('models', MODEL_KEYS_TO_RUN, 'limit', LIMIT)


In [ ]:
cache = pd.read_parquet(DATASET / 'tiki_translation_cache.parquet')
meta = pd.read_parquet(DATASET / 'tiki_bt_input.parquet')
meta = meta[meta['bt_route'] == 'mt'].drop_duplicates('cache_key')
meta_cols = [c for c in ['cache_key', 'expected_zh_value_chars', 'zh_value_chars_p05', 'zh_value_chars_p95', 'length_prior_soft_qc'] if c in meta.columns]
df = cache[cache['bt_route'] == 'mt'].merge(meta[meta_cols], on='cache_key', how='left')
df['source_chars'] = df['source_text_bt'].fillna('').str.replace(' ', '', regex=False).str.len()
if LIMIT:
    n = min(int(LIMIT), len(df))
    df = df.sample(n=n, random_state=42).reset_index(drop=True)
else:
    df = df.sort_values(['min_priority', 'source_chars']).reset_index(drop=True)
print('MT cache rows', len(df))
print(df[['cache_key','source_text_bt','key_vi','value_class','min_priority']].head(3).to_string(index=False))


In [ ]:
SYSTEM_PROMPT = '''You are a professional Vietnamese-to-Simplified Chinese e-commerce translator.
Translate only the requested source text. Preserve brands, model codes, numbers and units.
Return JSON only: {"translation": "..."}
Do not explain, do not use markdown, do not translate the key itself.'''

def make_prompt(row):
    payload = {
        'task': 'translate_title' if not str(row.get('key_vi') or '').strip() else 'translate_value',
        'source_text': str(row.get('source_text_bt') or ''),
        'key_vi': str(row.get('key_vi') or ''),
        'key_zh': str(row.get('key_zh_effective') or ''),
        'value_class': str(row.get('value_class') or ''),
        'expected_zh_chars': int(row.get('expected_zh_value_chars') or 0),
    }
    return json.dumps(payload, ensure_ascii=False)

def parse_translation(text):
    text = (text or '').strip()
    if text.startswith('```'):
        text = text.replace('```json','',1).replace('```','').strip()
    start, end = text.find('{'), text.rfind('}')
    if start >= 0 and end > start:
        try:
            obj = json.loads(text[start:end+1])
            if isinstance(obj, dict) and obj.get('translation'):
                return str(obj['translation']).strip()
        except json.JSONDecodeError:
            pass
    return text.strip()


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def find_config(model_key):
    config_name = CONFIG_FILES.get(model_key, model_key.replace('-', '_') + '.json')
    for base in (DATASET, DATASET / 'configs', Path('/kaggle/input')):
        path = base / config_name
        if path.exists():
            return path
    matches = glob.glob(f'/kaggle/input/**/{config_name}', recursive=True)
    if matches:
        return Path(matches[0])
    raise FileNotFoundError(f'Không thấy config {config_name} cho {model_key}')

def load_config(model_key):
    path = find_config(model_key)
    cfg = json.loads(path.read_text(encoding='utf-8'))
    cfg['_config_path'] = str(path)
    return cfg

def resolve_model_path(model_key):
    source = MODEL_SOURCES[model_key]
    try:
        return kagglehub.model_download(source)
    except Exception as error:
        token = model_key.split('-')[-1].lower()
        candidates = list(Path('/kaggle/input').rglob('config.json'))
        for cfg in candidates:
            if token in str(cfg).lower():
                print('fallback model path', cfg.parent, 'after', error)
                return str(cfg.parent)
        raise

def run_model(model_key):
    cfg = load_config(model_key)
    model_id = cfg['model_id']
    out_dir = OUTPUT_ROOT / model_key
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / 'predictions.parquet'
    state_file = out_dir / 'run_state.json'

    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
    if model_key == 'qwen3-8b' and vram_gb < 40:
        print(f'SKIP {model_key}: cần >=40GB VRAM, hiện có {vram_gb}GB')
        return
    if model_key == 'qwen3-4b' and vram_gb < 16:
        print(f'SKIP {model_key}: cần >=16GB VRAM, hiện có {vram_gb}GB')
        return

    done = set()
    existing = pd.DataFrame()
    if out_file.exists():
        existing = pd.read_parquet(out_file)
        done = set(existing['cache_key'].astype(str))
        print(f'{model_key}: resume {len(done):,} done')

    todo = df[~df['cache_key'].astype(str).isin(done)].copy()
    print(f'{model_key}: todo {len(todo):,}/{len(df):,}; config={cfg["_config_path"]}')

    print('loading', model_id, 'source', MODEL_SOURCES.get(model_key))
    model_path = resolve_model_path(model_key)
    print('model_path', model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, local_files_only=True)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
        local_files_only=True,
    )
    model.eval()
    model.generation_config.pad_token_id = tokenizer.pad_token_id

    batch_size = int(cfg.get('batch_size', 8))
    if vram_gb < 20:
        batch_size = min(batch_size, 2)
    max_input_tokens = int(cfg.get('max_input_tokens', 1024))
    max_new_tokens = int(cfg.get('max_new_tokens', 128))
    records = []
    started = time.time()

    for start in range(0, len(todo), batch_size):
        batch = todo.iloc[start:start+batch_size]
        prompts = []
        for row in batch.to_dict('records'):
            messages = [
                {'role':'system','content':SYSTEM_PROMPT},
                {'role':'user','content':make_prompt(row)},
            ]
            try:
                text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True,
                    enable_thinking=bool(cfg.get('enable_thinking', False)),
                )
            except TypeError:
                text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True,
                )
            prompts.append(text)

        enc = tokenizer(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=max_input_tokens,
        ).to(model.device)

        t0 = time.time()
        with torch.inference_mode():
            output = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=int(cfg.get('num_beams', 1)),
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
        latency = (time.time() - t0) * 1000
        generated = output[:, enc['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True)

        for row, raw in zip(batch.to_dict('records'), texts):
            records.append({
                'cache_key': str(row['cache_key']),
                'source_text_bt': row.get('source_text_bt'),
                'key_vi': row.get('key_vi'),
                'key_zh_effective': row.get('key_zh_effective'),
                'value_class': row.get('value_class'),
                'target_text_bt': parse_translation(raw),
                'raw_output': raw,
                'model_id': model_id,
                'prompt_version': PROMPT_VERSION,
                'qc_status': 'pending',
                'latency_ms': round(latency / max(1, len(batch)), 2),
            })

        if len(records) >= CHECKPOINT_EVERY or start + batch_size >= len(todo):
            part = pd.DataFrame(records)
            merged = pd.concat([existing, part], ignore_index=True) if len(existing) else part
            merged.to_parquet(out_file, index=False)
            existing = merged
            state_file.write_text(json.dumps({
                'model_key': model_key,
                'model_id': model_id,
                'n_done': len(merged),
                'n_total': len(df),
                'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            }, ensure_ascii=False, indent=2), encoding='utf-8')
            print(f'{model_key}: saved {len(merged):,} / {len(df):,}')
            records = []

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(model_key, 'done in', round((time.time()-started)/60, 2), 'min')


In [ ]:
for model_key in MODEL_KEYS_TO_RUN:
    run_model(model_key)


In [ ]:
print('Outputs:')
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file():
        print(path, path.stat().st_size)
